# Fraud Detection ML Pipeline - Setup

This notebook sets up the infrastructure for our fraud detection ML pipeline:
- Database and schema creation
- Warehouse configuration
- Synthetic transaction data generation
- Customer profile data creation
- Feature Store initialization and entity registration

### Import Required Libraries

We import Snowpark for data manipulation, Feature Store APIs for ML feature management, and our custom utility functions that handle synthetic data generation. These utilities encapsulate the logic for creating realistic fraud patterns in our test data.

In [ ]:
import sys
sys.path.append('../source')

from snowflake.snowpark import Session
from snowflake.snowpark.types import *
from snowflake.snowpark import functions as F
from snowflake.ml.feature_store import FeatureStore, Entity, FeatureView, CreationMode
import pandas as pd
import numpy as np

from utils import (
    generate_synthetic_fraud_data,
    generate_customer_profiles
)

### Initialize Snowpark Session

Get the active Snowpark session from the notebook environment. This session provides authenticated access to Snowflake and is used for all subsequent database operations.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()

session.sql("USE ROLE ACCOUNTADMIN").collect()
print(f"Role set to: {session.get_current_role()}")

## Configuration

### Define Project Constants

Centralize all configuration in one place for easy modification. Key settings:
- **N_TRANSACTIONS**: 100K transactions provides enough data for meaningful ML training
- **FRAUD_RATIO**: 2% fraud rate reflects real-world class imbalance (fraud is rare)
- **Separate schemas** for ML pipeline objects vs Feature Store to maintain organization

In [ ]:
DATABASE = "DEMO_FRAUD_DETECTION_DB"
SCHEMA = "ML_PIPELINE"
FEATURE_STORE_SCHEMA = "FEATURE_STORE"
WAREHOUSE = "ML_WH"

TRANSACTIONS_TABLE = "TRANSACTIONS"
CUSTOMER_PROFILES_TABLE = "CUSTOMER_PROFILES"
TRAINING_DATA_TABLE = "FRAUD_TRAINING_DATA"

N_TRANSACTIONS = 100000
FRAUD_RATIO = 0.02

## Create Infrastructure

### Create Database, Schemas, and Set Context

Establish the Snowflake infrastructure needed for the ML pipeline. We create:
- A dedicated **database** to isolate fraud detection assets
- Separate **schemas** for pipeline objects and Feature Store
- Set the session context so all subsequent operations use these defaults

**Why this matters**: Proper organization enables access control, cost tracking, and prevents conflicts with other projects.

In [ ]:
session.sql(f"CREATE DATABASE IF NOT EXISTS {DATABASE}").collect()
session.sql(f"CREATE SCHEMA IF NOT EXISTS {DATABASE}.{SCHEMA}").collect()
session.sql(f"CREATE SCHEMA IF NOT EXISTS {DATABASE}.{FEATURE_STORE_SCHEMA}").collect()
session.sql(f"USE DATABASE {DATABASE}").collect()
session.sql(f"USE SCHEMA {SCHEMA}").collect()
session.sql(f"USE WAREHOUSE {WAREHOUSE}").collect()
print(f"Infrastructure created: {DATABASE}.{SCHEMA}")

## Generate Synthetic Fraud Data

### Generate Transaction Data with Realistic Fraud Patterns

Create synthetic transaction data that mimics real-world fraud patterns:
- **High-value transactions**: Fraudsters often make large purchases
- **Odd-hour transactions**: Fraud often occurs at unusual times (2-5 AM)
- **International transactions**: Cross-border fraud is common
- **Card-not-present**: Online/phone fraud without physical card

**Why synthetic data?** It allows us to control fraud patterns, ensure privacy, and create a balanced dataset for model training.

In [ ]:
print(f"Generating {N_TRANSACTIONS:,} synthetic transactions with {FRAUD_RATIO*100:.1f}% fraud rate...")
transactions_df = generate_synthetic_fraud_data(
    n_transactions=N_TRANSACTIONS,
    fraud_ratio=FRAUD_RATIO,
    seed=42
)

print(f"\nTransaction data shape: {transactions_df.shape}")
print(f"Fraud distribution:\n{transactions_df['IS_FRAUD'].value_counts()}")
print(f"\nSample transactions:")
transactions_df.head()

### Generate Customer Profile Data

Create customer profiles with attributes that influence fraud risk:
- **Account age**: Newer accounts have higher fraud risk
- **Credit score**: Lower scores may correlate with fraud
- **Previous fraud reports**: History of fraud indicates higher risk
- **Premium membership**: May affect spending patterns

**Why this matters**: Customer context is crucial for accurate fraud detection - the same transaction may be normal for one customer but suspicious for another.

In [ ]:
unique_customers = transactions_df["CUSTOMER_ID"].unique().tolist()
print(f"Generating profiles for {len(unique_customers):,} customers...")

customer_profiles_df = generate_customer_profiles(unique_customers, seed=42)
print(f"Customer profiles shape: {customer_profiles_df.shape}")
customer_profiles_df.head()

## Load Data to Snowflake

### Load Transaction Data to Snowflake

Persist the transaction data as a Snowflake table. Using `mode("overwrite")` ensures idempotent execution - running this notebook multiple times produces the same result.

**Why Snowflake tables?** Enables distributed processing, integrates with Feature Store, and provides governance/lineage tracking.

In [ ]:
print("Loading transactions to Snowflake...")
transactions_spdf = session.create_dataframe(transactions_df)
transactions_spdf.write.mode("overwrite").save_as_table(f"{DATABASE}.{SCHEMA}.{TRANSACTIONS_TABLE}")
print(f"Loaded {session.table(f'{DATABASE}.{SCHEMA}.{TRANSACTIONS_TABLE}').count():,} transactions")

### Load Customer Profiles to Snowflake

Store customer profiles as a separate table. This dimensional design allows:
- Efficient joins with transaction data
- Independent updates to customer information
- Reuse across multiple ML models

In [ ]:
print("Loading customer profiles to Snowflake...")
profiles_spdf = session.create_dataframe(customer_profiles_df)
profiles_spdf.write.mode("overwrite").save_as_table(f"{DATABASE}.{SCHEMA}.{CUSTOMER_PROFILES_TABLE}")
print(f"Loaded {session.table(f'{DATABASE}.{SCHEMA}.{CUSTOMER_PROFILES_TABLE}').count():,} customer profiles")

## Initialize Feature Store

- **Feature Store**: The top-level container. You create/connect to one via FeatureStore(session, database, name, ...).

- **Entity**: Organizes feature views by subject (e.g., "transactions", "customer profile"). Entities define the primary keys for feature lookups.

- **Feature View**: Encapsulates the transformation of raw data into features. 
Two types:
    - **Snowflake-managed**: Backed by a dynamic table that auto-refreshes on a schedule (refresh_freq).

    - **External**: Backed by a view on a table you maintain yourself (e.g., via dbt). refresh_freq=None.

- **Feature Table**: The underlying storage (dynamic table or view) that holds the computed feature values.

-  **Dataset**:Generated via `fs.generate_dataset()` for training. Produces a point-in-time correct snapshot of features joined to your spine/label data.

### Create Feature Store

In [ ]:
fs = FeatureStore(
    session=session,
    database=DATABASE,
    name=FEATURE_STORE_SCHEMA,
    default_warehouse=WAREHOUSE,
    creation_mode=CreationMode.CREATE_IF_NOT_EXIST
)
print(f"Feature Store initialized: {DATABASE}.{FEATURE_STORE_SCHEMA}")

### Register Customer Entity

- Uses **CUSTOMER_ID** as the join key
- Links customer profile and aggregated features together

In [ ]:
customer_entity = Entity(
    name="CUSTOMER",
    join_keys=["CUSTOMER_ID"],
    desc="Customer entity for fraud detection features"
)
fs.register_entity(customer_entity)
print("Customer entity registered")

### Register Transaction Entity

- Uses **TRANSACTION_ID** as the join key
- Links aggregated transactions features and customer profile 

In [ ]:
transaction_entity = Entity(
    name="TRANSACTION",
    join_keys=["TRANSACTION_ID"],
    desc="Transaction entity for fraud detection"
)
fs.register_entity(transaction_entity)
print("Transaction entity registered")

## Verify Setup

### Validate Infrastructure and Data

Verification is crucial before proceeding to ensure:
- All tables were created successfully
- Data volumes match expectations
- Fraud ratio is as configured (~2%)
- Feature Store entities are registered


In [ ]:
print("\n=== Setup Verification ===")
print(f"\nTables created:")
tables = session.sql(f"SHOW TABLES IN {DATABASE}.{SCHEMA}").collect()
for t in tables:
    print(f"  - {t['name']}")

print(f"\nEntities in Feature Store:")
for entity in fs.list_entities().collect():
    print(f"  - {entity['NAME']}")

print(f"\nData Statistics:")
txn_count = session.table(f"{DATABASE}.{SCHEMA}.{TRANSACTIONS_TABLE}").count()
fraud_count = session.table(f"{DATABASE}.{SCHEMA}.{TRANSACTIONS_TABLE}").filter(F.col("IS_FRAUD") == 1).count()
print(f"  - Total transactions: {txn_count:,}")
print(f"  - Fraudulent transactions: {fraud_count:,} ({fraud_count/txn_count*100:.2f}%)")
print(f"  - Unique customers: {session.table(f'{DATABASE}.{SCHEMA}.{CUSTOMER_PROFILES_TABLE}').count():,}")